# Plugins: Bundling Hooks, Tools, and State

This tutorial introduces the Strands Agents `Plugin` abstraction. A plugin is a single Python class that bundles `@hook`-decorated lifecycle callbacks, `@tool`-decorated methods, and any plugin-resident state into one installable unit you attach to an agent via `Agent(plugins=[...])`. By the end you will know how to lift any existing `HookProvider` into a `Plugin`, how to expose hook-collected state as a tool the agent itself can call, how to reason about isolated vs. shared plugin instances, and how registration order drives middleware-style composition.


## Section 0 — Setup

We define one model and two simple tools here. Every later section reuses them.


In [ ]:
%pip install -r requirements.txt --quiet

In [ ]:
import logging

from strands import Agent, tool
from strands.models import BedrockModel
from strands.hooks import (
    HookProvider,
    HookRegistry,
    BeforeInvocationEvent,
    AfterInvocationEvent,
    BeforeModelCallEvent,
    AfterModelCallEvent,
    BeforeToolCallEvent,
    AfterToolCallEvent,
)
from strands.plugins import Plugin, hook

# Match the inference profile used by neighboring 01-learn tutorials.
MODEL_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"

# Quiet SDK logs so plugin behavior stays the focus of the output.
logging.getLogger("strands").setLevel(logging.WARNING)

We use Claude Haiku 4.5 on Amazon Bedrock to match neighboring 01-learn tutorials. Any Strands-supported model will work — swap the `model=` argument in the `Agent(...)` constructor if you prefer a different provider (see [Model Providers](https://strandsagents.com/docs/user-guide/concepts/model-providers/)).

In [ ]:
model = BedrockModel(model_id=MODEL_ID)

In [ ]:
@tool
def get_weather(city: str) -> str:
    """Return the current weather for a city."""
    fake = {"Seattle": "rainy, 52F", "Austin": "sunny, 78F"}
    return fake.get(city, "clear, 70F")


@tool
def lookup_user(user_id: str) -> dict:
    """Return the profile for a user id."""
    fake = {"U-42": {"name": "Ada", "role": "engineer"}}
    return fake.get(user_id, {"name": "unknown", "role": "unknown"})

## Section 1 — Raw hooks → Plugin transition

If you have used `HookProvider` (see [13-human-in-the-loop](../13-human-in-the-loop/) and [16-hooks-lifecycle](../16-hooks-lifecycle/)), the `Plugin` class is mostly the same idea with friendlier ergonomics. We will write the same observable behavior twice — once as a raw `HookProvider`, once as a `Plugin` with `@hook`-decorated methods — and then assert the two produce an identical hook firing sequence.

### 1a. The HookProvider version

`RawLoggingHooks` registers two callbacks imperatively, capturing each event firing as a `(event_type_name, phase)` tuple onto a list it owns.

In [ ]:
class RawLoggingHooks(HookProvider):
    """HookProvider equivalent of LoggingPlugin, for the side-by-side."""

    def __init__(self) -> None:
        self.events: list[tuple[str, str]] = []

    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(BeforeModelCallEvent, self._before_model)
        registry.add_callback(AfterModelCallEvent, self._after_model)

    def _before_model(self, event: BeforeModelCallEvent) -> None:
        self.events.append(("BeforeModelCallEvent", "before"))

    def _after_model(self, event: AfterModelCallEvent) -> None:
        self.events.append(("AfterModelCallEvent", "after"))

### 1b. The Plugin version

`LoggingPlugin` does the same job declaratively: it subclasses `Plugin`, declares a stable `name`, and decorates each callback with `@hook`. The decorator infers the event type from the method's parameter annotation, so there is no `register_hooks` boilerplate.

In [ ]:
class LoggingPlugin(Plugin):
    """Records lifecycle events as (event_type_name, phase) tuples."""

    name = "logging"

    def __init__(self) -> None:
        super().__init__()
        self.events: list[tuple[str, str]] = []

    @hook
    def on_before_model(self, event: BeforeModelCallEvent) -> None:
        self.events.append(("BeforeModelCallEvent", "before"))

    @hook
    def on_after_model(self, event: AfterModelCallEvent) -> None:
        self.events.append(("AfterModelCallEvent", "after"))

**Aside — union-type `@hook` annotations.** When a single method handles more than one event, annotate the parameter with a union and `@hook` will dispatch the method on each member of the union. The example below is illustrative only; it is not used in the equivalence assertion that follows.

In [ ]:
class _UnionDemo(Plugin):
    """Illustrates union-type @hook dispatch. Not used in the equivalence test."""

    name = "union-demo"

    @hook
    def on_model_event(
        self, event: BeforeModelCallEvent | AfterModelCallEvent
    ) -> None:
        # @hook registers this method for both event types.
        pass

### Equivalence test

Both classes capture the same `(event_type_name, phase)` tuples for the same set of events. If we run identical agents — same model, same tools, same prompt — through both, the captured sequences should match exactly. The assertion is on the captured event sequence, not on the model's natural-language reply (which is non-deterministic).

In [ ]:
raw = RawLoggingHooks()
plugin = LoggingPlugin()

raw_agent = Agent(
    model=model,
    tools=[get_weather, lookup_user],
    hooks=[raw],
)
plugin_agent = Agent(
    model=model,
    tools=[get_weather, lookup_user],
    plugins=[plugin],
)

PROMPT = "Look up the weather in Seattle."
raw_agent(PROMPT)
plugin_agent(PROMPT)

print("raw:   ", raw.events)
print("plugin:", plugin.events)

assert raw.events == plugin.events, (
    f"Hook firing sequences differ.\n"
    f"  raw:    {raw.events}\n"
    f"  plugin: {plugin.events}"
)

print(
    f"\nBoth versions captured the same {len(plugin.events)} hook firings in"
    " the same order. So we got identical observable behavior — the only thing"
    " that changed is how we wrote it down. The Plugin version drops the"
    " register_hooks boilerplate and uses @hook on the methods directly, but"
    " the runtime sees no difference."
)

### What changed

| Aspect | `HookProvider` (raw) | `Plugin` |
|---|---|---|
| Registration mechanism | Imperative `register_hooks(registry)` + `registry.add_callback(EventType, fn)` | Declarative `@hook` decorator on instance methods |
| Event subscription | One `add_callback` call per event type | Inferred from each method's parameter annotation (supports unions) |
| State storage | Provider instance attributes | Plugin instance attributes (same idea, same lifetime) |
| Tools bundled | Separate `@tool`-decorated functions passed via `tools=[...]` | `@tool`-decorated methods on the plugin itself, auto-registered |
| Reusability | Pass the same provider to many agents via `hooks=[...]` | Pass the same plugin to many agents via `plugins=[...]`; identified by stable `name` |

> The plugin registry rejects duplicate plugins. Two plugins with the same `name` attached to one agent will raise a runtime error at attach time, which is how the SDK keeps logs and configuration unambiguous when many plugins are stacked.

## Section 2 — `@tool` inside a plugin

The aha moment of `Plugin` is that one class ships hooks plus the tool that exposes what those hooks observed. In this section we build `MetricsPlugin`: hooks count tool calls and errors; a `get_audit_metrics` tool returns those counts back to the agent so the agent can introspect its own activity.

In [ ]:
class MetricsPlugin(Plugin):
    """Counts tool calls and errors via hooks; exposes a @tool to read them."""

    name = "metrics"

    def __init__(self) -> None:
        super().__init__()
        self.tool_calls: dict[str, int] = {}
        self.tool_errors: dict[str, int] = {}

    @hook
    def on_before_tool(self, event: BeforeToolCallEvent) -> None:
        if event.cancel_tool:
            # An earlier plugin already cancelled this call; do not count it.
            return
        tool_name = event.tool_use["name"]
        self.tool_calls[tool_name] = self.tool_calls.get(tool_name, 0) + 1

    @hook
    def on_after_tool(self, event: AfterToolCallEvent) -> None:
        if getattr(event, "exception", None) is not None:
            tool_name = event.tool_use["name"]
            self.tool_errors[tool_name] = self.tool_errors.get(tool_name, 0) + 1

    @tool
    def get_audit_metrics(self) -> dict:
        """Return current per-tool call and error counters collected by this plugin."""
        return {
            "tool_calls": dict(self.tool_calls),
            "tool_errors": dict(self.tool_errors),
        }

**Hooks observe; tools expose.** The two `@hook` methods are passive — they watch tool-call events and update plugin state. The `@tool` method is active — the model can decide to call it like any other tool. Both ship inside the same `MetricsPlugin` class, so the agent ships with the audit capability already wired up. There is nothing to register separately.

Note the `if event.cancel_tool: return` guard inside `on_before_tool`. `event.cancel_tool` does not suppress later `BeforeToolCallEvent` handlers; it only tells the tool executor to skip the tool itself. A plugin that wants to participate cleanly in a stack with policy plugins (like `SecurityPlugin` in Section 4) needs that defensive check, otherwise it counts work that never ran.

In [ ]:
metrics = MetricsPlugin()
agent = Agent(
    model=model,
    tools=[get_weather, lookup_user],
    plugins=[metrics],
)

# 1) Drive two tool calls. Naming the desired action by intent ("look up the
#    weather in Seattle and then in Austin") is enough to get the model to call
#    get_weather twice in sequence.
agent("Look up the weather in Seattle and then in Austin.")
print(f"After the priming prompt, MetricsPlugin sees: {metrics.tool_calls}")
assert metrics.tool_calls.get("get_weather", 0) == 2, metrics.tool_calls
print(
    "Two tool calls in, two counted. The BeforeToolCallEvent hook on"
    " MetricsPlugin observed both invocations.\n"
)

# 2) Now the agent introspects its own observation state via the bundled tool.
print("--- Agent's reply when asked to call get_audit_metrics ---")
result = agent("Use your get_audit_metrics tool to report current tool-call counts.")
print(result)
print(f"\nFinal MetricsPlugin state: {metrics.tool_calls}")
assert metrics.tool_calls.get("get_audit_metrics", 0) >= 1, metrics.tool_calls
print(
    "And there is the aha moment: the agent just called the @tool that the"
    " plugin itself contributed, and got back the data the plugin's hooks had"
    " been quietly collecting. One class shipped both halves — the observation"
    " and the way to read it back — wired up by Strands automatically."
)

The agent has introspected its own observation state via a tool. That is the Plugin shape — observation and exposure shipped together as one unit.

## Section 3 — Plugin state and sharing

Plugin state is just instance state on a Python object. That gives you two natural sharing modes: each agent gets its own plugin instance (state is isolated), or several agents share one instance (state aggregates). We will demonstrate both, then close with a short table on when to reach for `agent.state` or an external store instead.

### 3a. Isolated instances

Each agent gets its own `MetricsPlugin()` instance. The two plugins do not share state — they are unrelated Python objects. Whatever Agent A does, only Agent A's plugin records.

In [ ]:
m_a = MetricsPlugin()
m_b = MetricsPlugin()

agent_a = Agent(model=model, tools=[get_weather, lookup_user], plugins=[m_a])
agent_b = Agent(model=model, tools=[get_weather, lookup_user], plugins=[m_b])

agent_a("Look up the weather in Seattle.")
agent_b("Look up user U-42.")

print("agent_a's plugin counts:", m_a.tool_calls)
print("agent_b's plugin counts:", m_b.tool_calls)

# Each plugin only saw the work of its own agent.
assert "get_weather" in m_a.tool_calls, m_a.tool_calls
assert "get_weather" not in m_b.tool_calls, m_b.tool_calls
assert "lookup_user" in m_b.tool_calls, m_b.tool_calls
assert "lookup_user" not in m_a.tool_calls, m_a.tool_calls

print(
    "\nNotice the two dicts are disjoint. Each agent has its own MetricsPlugin"
    " instance, so each one only sees the tool calls made by its own agent."
    " That is what isolated state looks like — nothing is leaking between agents."
)

### 3b. Shared instance

Pass the same plugin object to two agents and they share state. This is the canonical pattern for cross-agent observability — one metrics sink, many agents.

In [ ]:
shared = MetricsPlugin()

agent_a = Agent(model=model, tools=[get_weather, lookup_user], plugins=[shared])
agent_b = Agent(model=model, tools=[get_weather, lookup_user], plugins=[shared])

agent_a("Look up the weather in Seattle.")
agent_b("Look up user U-42.")

print("shared plugin counts:", shared.tool_calls)

# One plugin instance, both agents' work rolled up.
assert shared.tool_calls.get("get_weather", 0) >= 1, shared.tool_calls
assert shared.tool_calls.get("lookup_user", 0) >= 1, shared.tool_calls

print(
    "\nNow both tool names show up in the same dict. We passed the exact same"
    " plugin object to both agents, so both agents' hook firings landed on the"
    " same counters. This is the typical pattern when you want one metrics"
    " sink across many agents."
)

> **Concurrency caveat.** A shared plugin instance is shared mutable state. The pattern above is fine for the sequential, single-threaded notebook flow we have here. If your agents may execute concurrently (async, threads, multiple processes touching the same in-memory plugin), the hook bodies need their own synchronization — for example a lock around the counter updates, or atomic structures like `collections.Counter` guarded with `threading.Lock`. The `Plugin` base class does not add concurrency control for you.

### 3c. When to use `agent.state`

Plugin instance attributes are not the only place state can live. Two other options worth knowing:

- **`agent.state`** — a per-agent dict managed by the `Agent` itself. Survives serialization with the session; visible to other plugins on the same agent. Initialize it in `init_agent(self, agent)` if you want a default value present from construction.
- **An external store** — a database, a cache, a queue. Necessary when state must outlive the process or be read by other workers.

#### Where should this state live?

| State location | Use when |
|---|---|
| Plugin instance attribute (what we used above) | The state is observation collected by the plugin itself, possibly aggregated across agents. Process-local. |
| `agent.state` (`agent.state.set("k", v)`) | The state is scoped to one agent's lifetime, must serialize with the session, or is read by other plugins attached to that same agent. |
| External store (DB, Redis, file) | The state must outlive the process, be visible to other workers, or be queried by services outside the agent runtime. |

This section is descriptive only — we are not deploying or configuring any external state store. Pick the location that matches your data's lifetime and visibility requirements.

## Section 4 — Composing multiple plugins

Plugins compose. You attach a list — `plugins=[A, B, C]` — and Strands invokes their hooks in the order you registered them for `Before*` events, and in reverse order for `After*` events. That is the same try/finally invariant the raw hook system uses; nothing new to learn at the wiring level.

Hook ordering does **not** auto-short-circuit. If `SecurityPlugin.on_before_tool` sets `event.cancel_tool`, the tool will not run — but every other `BeforeToolCallEvent` handler still fires. Plugins that participate in a stack are expected to respect what earlier plugins did. That is why our `MetricsPlugin` checks `event.cancel_tool` before incrementing: when Security runs first, Metrics sees the cancelled state and skips counting. When Metrics runs first, the flag is not set yet and Metrics counts a call that will then get cancelled. Same plugins, same prompt — different counters.

In [ ]:
class SecurityPlugin(Plugin):
    """Blocks tool calls whose name is in the configured denylist."""

    name = "security"

    def __init__(self, blocked_tools: set[str]) -> None:
        super().__init__()
        self.blocked_tools = set(blocked_tools)
        self.blocks: list[str] = []

    @hook
    def on_before_tool(self, event: BeforeToolCallEvent) -> None:
        tool_name = event.tool_use["name"]
        if tool_name in self.blocked_tools:
            self.blocks.append(tool_name)
            event.cancel_tool = (
                f"Security policy blocks the {tool_name!r} tool. "
                "Answer with what you have."
            )

### Two registration orders, one prompt

We will run the same prompt — `"Look up user U-42."` — twice. Both runs use a `SecurityPlugin` configured to block `lookup_user`. The only difference is the position of `MetricsPlugin` in the registration list.

- **Order 1** `[Security, Logging, Metrics]` — Security runs first and sets `event.cancel_tool`. By the time `MetricsPlugin.on_before_tool` runs, that flag is set; Metrics's defensive check returns early and the counter stays at zero. This is the enterprise pattern: policy first, observability after.
- **Order 2** `[Metrics, Logging, Security]` — Metrics runs first. The cancel flag has not been set yet, so Metrics counts the call. Then Security runs and cancels. Now your observability is reporting work that did not actually happen.

Same plugins, same prompt — different counters, because of registration order plus the defensive `cancel_tool` check inside `MetricsPlugin.on_before_tool`.

In [ ]:
def run_with_plugin_order(plugins: list[Plugin]) -> tuple[MetricsPlugin, SecurityPlugin]:
    metrics = next(p for p in plugins if isinstance(p, MetricsPlugin))
    security = next(p for p in plugins if isinstance(p, SecurityPlugin))
    agent = Agent(
        model=model,
        tools=[get_weather, lookup_user],
        plugins=plugins,
    )
    agent("Look up user U-42.")
    return metrics, security


# Order 1 — policy first.
m1, s1 = run_with_plugin_order([
    SecurityPlugin(blocked_tools={"lookup_user"}),
    LoggingPlugin(),
    MetricsPlugin(),
])
print("Order 1 [Security, Logging, Metrics]")
print(f"  metrics counts:  {m1.tool_calls}")
print(f"  security blocks: {s1.blocks}")
assert m1.tool_calls.get("lookup_user", 0) == 0, m1.tool_calls
assert "lookup_user" in s1.blocks, s1.blocks
print(
    "  Security ran first and set event.cancel_tool. By the time Metrics's"
    " hook ran, that flag was already set, so its defensive check skipped the"
    " increment. The lookup_user counter is zero — which is the truth, because"
    " the tool never actually ran.\n"
)

# Order 2 — observability before policy.
m2, s2 = run_with_plugin_order([
    MetricsPlugin(),
    LoggingPlugin(),
    SecurityPlugin(blocked_tools={"lookup_user"}),
])
print("Order 2 [Metrics, Logging, Security]")
print(f"  metrics counts:  {m2.tool_calls}")
print(f"  security blocks: {s2.blocks}")
assert m2.tool_calls.get("lookup_user", 0) >= 1, m2.tool_calls
assert "lookup_user" in s2.blocks, s2.blocks
print(
    "  Metrics ran first this time. cancel_tool was not set yet, so it counted"
    " a call that Security then blocked a moment later. Same plugins, same"
    " prompt, but the dashboard now shows work that did not actually happen."
    " That is why production stacks put policy ahead of observability."
)

The takeaway is the enterprise middleware pattern: stack `Security → Logging → Metrics` so that policy decisions short-circuit before observability records anything, and so the surrounding logger sees both what was asked and what was blocked. `Before*` handlers fire in registration order; `After*` handlers fire in reverse, so the outer plugin's `After*` always sees what the inner plugins did.

### Recap

| Concept | Raw hooks | Plugin |
|---|---|---|
| Registration API | `register_hooks(registry)` + `registry.add_callback(EventType, fn)` | `class P(Plugin)` with `name = "..."` and `@hook` methods |
| Event subscription | One `add_callback` call per event type | Inferred from each method's parameter annotation; supports unions |
| Bundling tools | Tools live outside the hook class, passed via `Agent(tools=[...])` | Tools live inside the plugin as `@tool`-decorated methods, auto-registered |
| Where state lives | Provider instance attributes | Plugin instance attributes (with `agent.state` and external stores as alternatives) |
| Composition order | List passed to `hooks=[...]` | List passed to `plugins=[...]`; same registration-order semantics, plus a stable `name` per plugin |

### Where to next

- [`15-skills`](../15-skills/) is a real-world plugin you can install today (`AgentSkills`). Now that you have built a plugin from scratch, the source of `AgentSkills` should read clearly.
- [`16-hooks-lifecycle`](../16-hooks-lifecycle/) is the full event catalogue, including writable fields like `cancel_tool`, `retry`, and `resume` that we did not exercise in every plugin here.

One last hook worth knowing about, even though no plugin in this notebook used it: `init_agent(self, agent)` runs once when the plugin is attached to an agent. It is the right place for one-time setup that needs the constructed `Agent` (initializing `agent.state` defaults, opening connections, registering tools conditionally). See the [Plugin documentation](https://strandsagents.com/docs/user-guide/concepts/plugins/) for the full method signature.